# Root Cause Analysis Notebook

This notebook reproduces the updated leakage-safe root cause analysis for the clean point-level manufacturing dataset.

It performs:
- side-by-side comparison for `pre_any_defect_50m`, `pre_any_defect_200m`, and `any_defect`
- interpretable modeling with logistic regression and random forest
- single-defect rerun for defect type 3 plus grouped variable/system importance
- driver extraction, interaction analysis, target comparison tables, and segment volatility checks
- export of the business-facing markdown report and supporting CSV/PNG outputs to `rca_outputs/`


In [ ]:
from __future__ import annotations

import json
import math
import os
import shutil
import sys
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable

BASE_DIR = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()

VENDOR_DIR = BASE_DIR / ".vendor"
if VENDOR_DIR.exists():
    sys.path.insert(0, str(VENDOR_DIR))

MPL_DIR = BASE_DIR / ".mplconfig"
os.environ.setdefault("MPLCONFIGDIR", str(MPL_DIR))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


RANDOM_STATE = 42
GENERAL_TARGET_ORDER = ["pre_any_defect_50m", "pre_any_defect_200m", "any_defect"]
LABEL_COLS = [
    "any_defect",
    "pre_any_defect_50m",
    "pre_any_defect_200m",
    "pre_any_defect_500m",
]
TYPE3_LABEL_COLS = [
    "defect_type3_any_defect",
    "defect_type3_pre_any_defect_50m",
    "defect_type3_pre_any_defect_200m",
    "defect_type3_related_500m",
]
ID_COLS = ["COIL", "DATE", "MT", "TIME_START_PROCESS"]
EXCLUDED_FEATURES = set(ID_COLS + LABEL_COLS + TYPE3_LABEL_COLS + ["run_id"])
OUTPUT_DIR = BASE_DIR / "rca_outputs"
DATA_PATH = BASE_DIR / "rca_point_level_clean.csv"
VALIDATION_PATH = BASE_DIR / "rca_build_validation.json"
AUDIT_PATH = BASE_DIR / "rca_defect_coverage_audit.csv"

TARGET_DESCRIPTIONS = {
    "pre_any_defect_50m": {
        "display_name": "Pre-defect 50m",
        "capture_text": "Measurements in the last 50 m before a labeled defect, excluding active defect rows.",
        "operational_value": "Best for near-term intervention when operators need the strongest upstream signal.",
    },
    "pre_any_defect_200m": {
        "display_name": "Pre-defect 200m",
        "capture_text": "Measurements in the broader 200 m run-up before a labeled defect, excluding active defect rows.",
        "operational_value": "Best for earlier monitoring when operations need more lead time.",
    },
    "any_defect": {
        "display_name": "In-defect",
        "capture_text": "Measurements inside the labeled defect region, which mix upstream drivers with failure-state symptoms.",
        "operational_value": "Useful as a contrast target to separate warning signals from symptom signals.",
    },
    "defect_type3_pre_any_defect_50m": {
        "display_name": "Type 3 Pre-defect 50m",
        "capture_text": "Measurements in the last 50 m before defect type 3 intervals only.",
        "operational_value": "Useful for checking whether one defect family has a clearer or narrower warning signature.",
    },
    "defect_type3_any_defect": {
        "display_name": "Type 3 In-defect",
        "capture_text": "Measurements inside defect type 3 intervals only.",
        "operational_value": "Useful for contrasting type 3 failure-state signals against the type 3 warning window.",
    },
}


@dataclass(frozen=True)
class TargetSpec:
    target: str
    output_prefix: str
    scope: str
    active_exclusion_col: str | None = None


@dataclass
class ModelArtifacts:
    logistic: Pipeline
    forest: Pipeline
    metrics: dict[str, float]
    logistic_table: pd.DataFrame
    forest_importance: pd.DataFrame
    driver_table: pd.DataFrame
    risky_pairs: pd.DataFrame
    top_rules: pd.DataFrame


@dataclass
class TargetAnalysis:
    spec: TargetSpec
    df: pd.DataFrame
    train_df: pd.DataFrame
    test_df: pd.DataFrame
    correlation_full: pd.DataFrame
    correlation_top: pd.DataFrame
    artifacts: ModelArtifacts

    @property
    def display_name(self) -> str:
        return TARGET_DESCRIPTIONS[self.spec.target]["display_name"]

    @property
    def population_rows(self) -> int:
        return len(self.df)

    @property
    def positive_rows(self) -> int:
        return int(self.df[self.spec.target].sum())

    @property
    def positive_rate(self) -> float:
        return float(self.df[self.spec.target].mean())


GENERAL_SPECS = [
    TargetSpec(target="pre_any_defect_50m", output_prefix="pre_any_defect_50m", scope="general", active_exclusion_col="any_defect"),
    TargetSpec(target="pre_any_defect_200m", output_prefix="pre_any_defect_200m", scope="general", active_exclusion_col="any_defect"),
    TargetSpec(target="any_defect", output_prefix="any_defect", scope="general"),
]

TYPE3_SPECS = [
    TargetSpec(
        target="defect_type3_pre_any_defect_50m",
        output_prefix="defect_type3_pre_any_defect_50m",
        scope="defect_type_3",
        active_exclusion_col="defect_type3_any_defect",
    ),
    TargetSpec(target="defect_type3_any_defect", output_prefix="defect_type3_any_defect", scope="defect_type_3"),
]


def pct(value: float) -> str:
    return f"{100 * value:.2f}%"


def ensure_output_dir() -> None:
    OUTPUT_DIR.mkdir(exist_ok=True)
    MPL_DIR.mkdir(exist_ok=True)


def is_early_warning_target(target: str) -> bool:
    return "pre_any_defect" in target


def load_dataset(path: Path) -> tuple[pd.DataFrame, list[str]]:
    df = pd.read_csv(path, low_memory=False)
    df["COIL"] = df["COIL"].astype("string").str.strip()
    df["DATE"] = pd.to_datetime(df["DATE"], errors="coerce").dt.strftime("%Y-%m-%d")
    df["run_id"] = df["COIL"].fillna("missing") + "|" + df["DATE"].fillna("missing")

    feature_cols = [col for col in df.columns if col not in EXCLUDED_FEATURES]
    df[feature_cols] = df[feature_cols].apply(pd.to_numeric, errors="coerce").astype("float32")

    valid_feature_cols: list[str] = []
    for col in feature_cols:
        series = df[col]
        if series.notna().sum() < 1000:
            continue
        if series.nunique(dropna=True) <= 1:
            continue
        valid_feature_cols.append(col)

    return df, valid_feature_cols


def grouped_split(df: pd.DataFrame, target: str, test_size: float = 0.25, max_attempts: int = 20) -> tuple[np.ndarray, np.ndarray]:
    for offset in range(max_attempts):
        splitter = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=RANDOM_STATE + offset)
        train_idx, test_idx = next(splitter.split(df, groups=df["run_id"]))
        y_train = df.iloc[train_idx][target]
        y_test = df.iloc[test_idx][target]
        if y_train.nunique() >= 2 and y_test.nunique() >= 2 and y_train.sum() > 0 and y_test.sum() > 0:
            return train_idx, test_idx
    raise ValueError(f"Could not create a grouped split with positives in both train and test for target {target}.")


def build_target_population(df: pd.DataFrame, spec: TargetSpec) -> pd.DataFrame:
    work = df.loc[df[spec.target].notna()].copy()
    if spec.active_exclusion_col is not None:
        work = work.loc[work[spec.active_exclusion_col] == 0].copy()
    work[spec.target] = work[spec.target].astype("int8")
    return work.reset_index(drop=True)


def compute_correlations(
    df: pd.DataFrame,
    feature_cols: list[str],
    target: str,
    output_prefix: str,
    top_n: int = 15,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    y = df[target]
    records: list[dict[str, float | str]] = []
    for col in feature_cols:
        series = df[col]
        corr = series.corr(y)
        grouped = df.groupby(target, observed=True)[col]
        mean_by_class = grouped.mean()
        std_by_class = grouped.std()
        mean_0 = float(mean_by_class.get(0, np.nan))
        mean_1 = float(mean_by_class.get(1, np.nan))
        std_0 = float(std_by_class.get(0, np.nan))
        std_1 = float(std_by_class.get(1, np.nan))
        pooled_std = math.sqrt(np.nanmean([std_0**2, std_1**2])) if np.isfinite(std_0) or np.isfinite(std_1) else np.nan
        smd = (mean_1 - mean_0) / pooled_std if pooled_std and np.isfinite(pooled_std) and pooled_std > 0 else np.nan
        records.append(
            {
                "feature": col,
                "correlation": corr,
                "abs_correlation": abs(corr) if pd.notna(corr) else np.nan,
                "mean_normal": mean_0,
                "mean_positive": mean_1,
                "mean_delta": mean_1 - mean_0,
                "standardized_mean_diff": smd,
            }
        )

    full = (
        pd.DataFrame(records)
        .sort_values(["abs_correlation", "standardized_mean_diff"], ascending=[False, False], na_position="last")
        .reset_index(drop=True)
    )
    top = full.head(top_n).copy().reset_index(drop=True)
    full.to_csv(OUTPUT_DIR / f"correlation_table_{output_prefix}.csv", index=False)
    top.to_csv(OUTPUT_DIR / f"top_correlations_{output_prefix}.csv", index=False)
    return full, top


def draw_correlation_plot(corr_table: pd.DataFrame, display_name: str, output_prefix: str) -> Path:
    fig, ax = plt.subplots(figsize=(10, 6))
    plot_df = corr_table.sort_values("correlation")
    colors = np.where(plot_df["correlation"] >= 0, "#b73a3a", "#2364aa")
    ax.barh(plot_df["feature"], plot_df["correlation"], color=colors)
    ax.set_title(f"Top Feature Correlations With {display_name}")
    ax.set_xlabel("Pearson correlation")
    ax.set_ylabel("")
    ax.axvline(0, color="black", linewidth=0.8)
    fig.tight_layout()
    path = OUTPUT_DIR / f"corr_{output_prefix}.png"
    fig.savefig(path, dpi=180, bbox_inches="tight")
    plt.close(fig)
    return path


def draw_distribution_plot(
    df: pd.DataFrame,
    feature_cols: list[str],
    target: str,
    display_name: str,
    output_prefix: str,
    top_n: int = 4,
    sample_per_class: int = 15000,
) -> Path:
    chosen = feature_cols[:top_n]
    pos = df.loc[df[target] == 1, chosen + [target]]
    neg = df.loc[df[target] == 0, chosen + [target]]
    if len(pos) > sample_per_class:
        pos = pos.sample(sample_per_class, random_state=RANDOM_STATE)
    if len(neg) > sample_per_class:
        neg = neg.sample(sample_per_class, random_state=RANDOM_STATE)

    plot_df = pd.concat([pos, neg], ignore_index=True)
    plot_df[target] = plot_df[target].map({0: "normal", 1: "positive"})

    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    axes = axes.flatten()
    for ax, col in zip(axes, chosen, strict=False):
        sns.histplot(
            data=plot_df,
            x=col,
            hue=target,
            bins=40,
            stat="density",
            common_norm=False,
            element="step",
            fill=False,
            ax=ax,
        )
        ax.set_title(f"{col} distribution")
        ax.set_ylabel("density")

    for ax in axes[len(chosen) :]:
        ax.axis("off")

    fig.suptitle(f"Positive vs normal distribution shifts for {display_name}", y=1.02, fontsize=14)
    fig.tight_layout()
    path = OUTPUT_DIR / f"dist_{output_prefix}.png"
    fig.savefig(path, dpi=180, bbox_inches="tight")
    plt.close(fig)
    return path


def lift_at_top_fraction(y_true: np.ndarray, y_score: np.ndarray, fraction: float = 0.05) -> float:
    order = np.argsort(y_score)[::-1]
    cutoff = max(1, int(len(order) * fraction))
    top_rate = y_true[order[:cutoff]].mean()
    base_rate = y_true.mean()
    return float(top_rate / base_rate) if base_rate > 0 else np.nan


def precision_recall_at_top_fraction(y_true: np.ndarray, y_score: np.ndarray, fraction: float = 0.05) -> tuple[float, float]:
    order = np.argsort(y_score)[::-1]
    cutoff = max(1, int(len(order) * fraction))
    top_idx = order[:cutoff]
    true_positives = float(y_true[top_idx].sum())
    precision = float(y_true[top_idx].mean())
    total_positives = float(y_true.sum())
    recall = true_positives / total_positives if total_positives > 0 else np.nan
    return precision, recall


def fit_models(
    X_train: pd.DataFrame,
    X_test: pd.DataFrame,
    y_train: pd.Series,
    y_test: pd.Series,
    feature_cols: list[str],
    target: str,
    output_prefix: str,
) -> ModelArtifacts:
    sparse_target = is_early_warning_target(target)
    logistic = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            (
                "logit",
                LogisticRegression(
                    penalty="l1",
                    solver="saga",
                    C=0.35 if sparse_target else 0.5,
                    class_weight="balanced",
                    max_iter=1800,
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    )

    forest = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            (
                "forest",
                RandomForestClassifier(
                    n_estimators=80,
                    max_depth=8,
                    min_samples_leaf=100 if sparse_target else 150,
                    class_weight="balanced_subsample",
                    max_features="sqrt",
                    n_jobs=1,
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    )

    logistic.fit(X_train, y_train)
    forest.fit(X_train, y_train)

    logit_pred = logistic.predict_proba(X_test)[:, 1]
    forest_pred = forest.predict_proba(X_test)[:, 1]

    logit_precision, logit_recall = precision_recall_at_top_fraction(y_test.to_numpy(), logit_pred, fraction=0.05)
    forest_precision, forest_recall = precision_recall_at_top_fraction(y_test.to_numpy(), forest_pred, fraction=0.05)

    metrics = {
        "target_rate_test": float(y_test.mean()),
        "logistic_roc_auc": float(roc_auc_score(y_test, logit_pred)),
        "logistic_pr_auc": float(average_precision_score(y_test, logit_pred)),
        "logistic_brier": float(brier_score_loss(y_test, logit_pred)),
        "logistic_lift_top_5pct": lift_at_top_fraction(y_test.to_numpy(), logit_pred, fraction=0.05),
        "logistic_precision_top_5pct": logit_precision,
        "logistic_recall_top_5pct": logit_recall,
        "forest_roc_auc": float(roc_auc_score(y_test, forest_pred)),
        "forest_pr_auc": float(average_precision_score(y_test, forest_pred)),
        "forest_brier": float(brier_score_loss(y_test, forest_pred)),
        "forest_lift_top_5pct": lift_at_top_fraction(y_test.to_numpy(), forest_pred, fraction=0.05),
        "forest_precision_top_5pct": forest_precision,
        "forest_recall_top_5pct": forest_recall,
    }

    logit_coef = logistic.named_steps["logit"].coef_[0]
    logistic_table = (
        pd.DataFrame(
            {
                "feature": feature_cols,
                "coefficient": logit_coef,
                "abs_coefficient": np.abs(logit_coef),
            }
        )
        .sort_values("abs_coefficient", ascending=False)
        .reset_index(drop=True)
    )
    logistic_table.to_csv(OUTPUT_DIR / f"logistic_coefficients_{output_prefix}.csv", index=False)

    pos_test = y_test[y_test == 1]
    neg_test = y_test[y_test == 0]
    sample_neg_n = min(len(neg_test), 8000 if sparse_target else 12000)
    sample_idx = pos_test.index.to_numpy()
    if sample_neg_n > 0:
        sampled_neg_idx = neg_test.sample(sample_neg_n, random_state=RANDOM_STATE).index.to_numpy()
        sample_idx = np.concatenate([sample_idx, sampled_neg_idx])
    if len(sample_idx) == 0:
        sample_idx = y_test.index.to_numpy()

    X_perm = X_test.loc[sample_idx]
    y_perm = y_test.loc[sample_idx]
    perm = permutation_importance(
        forest,
        X_perm,
        y_perm,
        n_repeats=1,
        random_state=RANDOM_STATE,
        scoring="average_precision" if sparse_target else "roc_auc",
        n_jobs=1,
    )
    forest_importance = (
        pd.DataFrame(
            {
                "feature": feature_cols,
                "importance_mean": perm.importances_mean,
                "importance_std": perm.importances_std,
            }
        )
        .sort_values("importance_mean", ascending=False)
        .reset_index(drop=True)
    )
    forest_importance.to_csv(OUTPUT_DIR / f"forest_permutation_importance_{output_prefix}.csv", index=False)

    driver_table = build_driver_table(
        X_train=X_train,
        X_test=X_test,
        y_test=y_test,
        logistic_table=logistic_table,
        forest_importance=forest_importance,
    )
    driver_table.to_csv(OUTPUT_DIR / f"driver_table_{output_prefix}.csv", index=False)

    risky_pairs = build_risky_pairs(X_train, X_test, y_test, driver_table, top_n=8)
    risky_pairs.to_csv(OUTPUT_DIR / f"risky_pairs_{output_prefix}.csv", index=False)

    top_rules = build_top_rules(X_train, X_test, y_train, y_test, driver_table, target)
    top_rules.to_csv(OUTPUT_DIR / f"top_rules_{output_prefix}.csv", index=False)

    return ModelArtifacts(
        logistic=logistic,
        forest=forest,
        metrics=metrics,
        logistic_table=logistic_table,
        forest_importance=forest_importance,
        driver_table=driver_table,
        risky_pairs=risky_pairs,
        top_rules=top_rules,
    )


def build_driver_table(
    X_train: pd.DataFrame,
    X_test: pd.DataFrame,
    y_test: pd.Series,
    logistic_table: pd.DataFrame,
    forest_importance: pd.DataFrame,
    top_n: int = 15,
) -> pd.DataFrame:
    work = logistic_table.merge(forest_importance, on="feature", how="outer")
    work["logistic_rank"] = work["abs_coefficient"].rank(ascending=False, method="min")
    work["forest_rank"] = work["importance_mean"].rank(ascending=False, method="min")
    work["consensus_rank"] = work[["logistic_rank", "forest_rank"]].mean(axis=1)

    directions: list[str] = []
    low_rates: list[float] = []
    high_rates: list[float] = []
    thresholds: list[str] = []
    lifts: list[float] = []
    mean_shifts: list[float] = []
    baseline_rate = float(y_test.mean())

    for feature in work["feature"]:
        train_series = X_train[feature]
        test_series = X_test[feature]
        q20 = float(train_series.quantile(0.2))
        q80 = float(train_series.quantile(0.8))
        low_mask = test_series <= q20
        high_mask = test_series >= q80
        low_rate = float(y_test[low_mask].mean()) if low_mask.any() else np.nan
        high_rate = float(y_test[high_mask].mean()) if high_mask.any() else np.nan
        mean_shift = float(test_series[y_test == 1].mean() - test_series[y_test == 0].mean())
        low_rates.append(low_rate)
        high_rates.append(high_rate)
        mean_shifts.append(mean_shift)

        if pd.notna(high_rate) and pd.notna(low_rate) and high_rate > low_rate:
            directions.append("higher is associated with higher risk")
            thresholds.append(f">= {q80:.3f}")
            lifts.append(high_rate / baseline_rate if baseline_rate > 0 else np.nan)
        else:
            directions.append("lower is associated with higher risk")
            thresholds.append(f"<= {q20:.3f}")
            lifts.append(low_rate / baseline_rate if baseline_rate > 0 else np.nan)

    work["risk_direction"] = directions
    work["low_quintile_rate"] = low_rates
    work["high_quintile_rate"] = high_rates
    work["quintile_threshold"] = thresholds
    work["risk_lift_vs_baseline"] = lifts
    work["holdout_mean_shift"] = mean_shifts

    return (
        work.sort_values(["consensus_rank", "risk_lift_vs_baseline"], ascending=[True, False])
        .head(top_n)
        .reset_index(drop=True)
    )


def build_risky_pairs(
    X_train: pd.DataFrame,
    X_test: pd.DataFrame,
    y_test: pd.Series,
    driver_table: pd.DataFrame,
    top_n: int = 8,
) -> pd.DataFrame:
    selected = driver_table.head(top_n).copy()
    base_rate = float(y_test.mean())
    rules: dict[str, tuple[str, float]] = {}
    for _, row in selected.iterrows():
        feature = row["feature"]
        if row["risk_direction"] == "higher is associated with higher risk":
            threshold = float(X_train[feature].quantile(0.8))
            rules[feature] = (">=", threshold)
        else:
            threshold = float(X_train[feature].quantile(0.2))
            rules[feature] = ("<=", threshold)

    records: list[dict[str, float | str]] = []
    features = list(rules)
    for idx, feature_a in enumerate(features):
        op_a, threshold_a = rules[feature_a]
        mask_a = X_test[feature_a] >= threshold_a if op_a == ">=" else X_test[feature_a] <= threshold_a
        for feature_b in features[idx + 1 :]:
            op_b, threshold_b = rules[feature_b]
            mask_b = X_test[feature_b] >= threshold_b if op_b == ">=" else X_test[feature_b] <= threshold_b
            mask = mask_a & mask_b
            support = int(mask.sum())
            if support < 150:
                continue
            rate = float(y_test[mask].mean())
            records.append(
                {
                    "rule_a": f"{feature_a} {op_a} {threshold_a:.3f}",
                    "rule_b": f"{feature_b} {op_b} {threshold_b:.3f}",
                    "support": support,
                    "positive_rate": rate,
                    "lift_vs_baseline": rate / base_rate if base_rate > 0 else np.nan,
                }
            )

    if not records:
        return pd.DataFrame(columns=["rule_a", "rule_b", "support", "positive_rate", "lift_vs_baseline"])

    return (
        pd.DataFrame(records)
        .sort_values(["lift_vs_baseline", "positive_rate", "support"], ascending=[False, False, False])
        .head(10)
        .reset_index(drop=True)
    )


def build_top_rules(
    X_train: pd.DataFrame,
    X_test: pd.DataFrame,
    y_train: pd.Series,
    y_test: pd.Series,
    driver_table: pd.DataFrame,
    target: str,
) -> pd.DataFrame:
    from sklearn.tree import DecisionTreeClassifier

    top_features = driver_table["feature"].head(8).tolist()
    if not top_features:
        return pd.DataFrame(columns=["rule", "support", "positive_rate", "lift_vs_baseline"])

    tree = DecisionTreeClassifier(
        max_depth=3,
        min_samples_leaf=180 if is_early_warning_target(target) else 260,
        class_weight="balanced",
        random_state=RANDOM_STATE,
    )
    imputer = SimpleImputer(strategy="median")
    X_train_imp = imputer.fit_transform(X_train[top_features])
    X_test_imp = imputer.transform(X_test[top_features])
    tree.fit(X_train_imp, y_train)

    node_features = tree.tree_.feature
    node_thresholds = tree.tree_.threshold
    children_left = tree.tree_.children_left
    children_right = tree.tree_.children_right
    test_leaves = tree.apply(X_test_imp)
    base_rate = float(y_test.mean())

    records: list[dict[str, float | str]] = []

    def walk(node_id: int, conditions: list[str]) -> None:
        left = children_left[node_id]
        right = children_right[node_id]
        if left == right:
            leaf_mask = test_leaves == node_id
            support = int(leaf_mask.sum())
            if support < 150:
                return
            rate = float(y_test[leaf_mask].mean())
            records.append(
                {
                    "rule": " AND ".join(conditions) if conditions else "all points",
                    "support": support,
                    "positive_rate": rate,
                    "lift_vs_baseline": rate / base_rate if base_rate > 0 else np.nan,
                }
            )
            return

        feature_name = top_features[node_features[node_id]]
        threshold = node_thresholds[node_id]
        walk(left, conditions + [f"{feature_name} <= {threshold:.3f}"])
        walk(right, conditions + [f"{feature_name} > {threshold:.3f}"])

    walk(0, [])

    if not records:
        return pd.DataFrame(columns=["rule", "support", "positive_rate", "lift_vs_baseline"])

    return (
        pd.DataFrame(records)
        .sort_values(["positive_rate", "support"], ascending=[False, False])
        .head(8)
        .reset_index(drop=True)
    )


def draw_importance_plot(driver_table: pd.DataFrame, display_name: str, output_prefix: str) -> Path:
    plot_df = driver_table.head(12).sort_values("consensus_rank", ascending=False)
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.barh(plot_df["feature"], plot_df["risk_lift_vs_baseline"], color="#ca6702")
    ax.set_title(f"Top Driver Lift Vs Baseline For {display_name}")
    ax.set_xlabel("Risk lift in risky quintile")
    ax.set_ylabel("")
    fig.tight_layout()
    path = OUTPUT_DIR / f"driver_lift_{output_prefix}.png"
    fig.savefig(path, dpi=180, bbox_inches="tight")
    plt.close(fig)
    return path


def draw_risk_curve_plot(
    X_test: pd.DataFrame,
    y_test: pd.Series,
    driver_table: pd.DataFrame,
    display_name: str,
    output_prefix: str,
    top_n: int = 6,
) -> Path:
    chosen = driver_table["feature"].head(top_n).tolist()
    fig, axes = plt.subplots(2, 3, figsize=(14, 8))
    axes = axes.flatten()
    for ax, feature in zip(axes, chosen, strict=False):
        temp = pd.DataFrame({"feature": X_test[feature], "target": y_test}).dropna()
        temp["bin"] = pd.qcut(temp["feature"], q=10, duplicates="drop")
        grouped = temp.groupby("bin", observed=True).agg(
            midpoint=("feature", "mean"),
            target_rate=("target", "mean"),
            count=("target", "size"),
        )
        ax.plot(grouped["midpoint"], grouped["target_rate"], marker="o", color="#005f73")
        ax.set_title(feature)
        ax.set_xlabel("feature value")
        ax.set_ylabel("positive rate")

    for ax in axes[len(chosen) :]:
        ax.axis("off")

    fig.suptitle(f"Marginal Risk Curves On Holdout Data For {display_name}", y=1.02, fontsize=14)
    fig.tight_layout()
    path = OUTPUT_DIR / f"risk_curves_{output_prefix}.png"
    fig.savefig(path, dpi=180, bbox_inches="tight")
    plt.close(fig)
    return path


def classify_feature_group(feature: str) -> str:
    if feature.startswith(("TEMP_", "PYRO_", "LASER_FRN_")) or feature in {"LS_OVEN"}:
        return "Thermal"
    if feature.startswith(("GAS_", "AIR_", "AIR_CH4_")):
        return "Combustion"
    if feature.startswith(("COOL_", "LASER_RAFF_")) or feature in {"LS_COOLING", "TOUT_RAFF_H2O", "N_RAMPE_H2O"}:
        return "Cooling"
    if feature.startswith(("TIRO_", "SPEED_", "FILL_")):
        return "Mechanical"
    if feature.startswith(("PRES_", "VENT_", "EXT_")) or "PRESSURE" in feature:
        return "Pressure and Ventilation"
    if feature.startswith(("DEC_", "ELET_")):
        return "Chemical and Pickling"
    if feature.startswith("SPZ"):
        return "Bridle and Roll Control"
    return "Other"


def build_grouped_importance(results: dict[str, TargetAnalysis]) -> pd.DataFrame:
    records: list[pd.DataFrame] = []
    for result in results.values():
        feature_frame = (
            result.artifacts.logistic_table.merge(result.artifacts.forest_importance, on="feature", how="outer")
            .merge(result.correlation_full[["feature", "abs_correlation"]], on="feature", how="left")
            .merge(result.artifacts.driver_table[["feature", "risk_lift_vs_baseline"]], on="feature", how="left")
        )
        feature_frame["system_group"] = feature_frame["feature"].map(classify_feature_group)
        feature_frame["logistic_share"] = feature_frame["abs_coefficient"] / feature_frame["abs_coefficient"].sum()
        forest_positive = feature_frame["importance_mean"].clip(lower=0)
        forest_total = forest_positive.sum()
        feature_frame["forest_share"] = forest_positive / forest_total if forest_total > 0 else 0.0
        corr_total = feature_frame["abs_correlation"].sum()
        feature_frame["correlation_share"] = feature_frame["abs_correlation"] / corr_total if corr_total > 0 else 0.0
        feature_frame["combined_feature_score"] = (
            feature_frame["logistic_share"].fillna(0)
            + feature_frame["forest_share"].fillna(0)
            + feature_frame["correlation_share"].fillna(0)
        ) / 3.0
        feature_frame["scope"] = result.spec.scope
        feature_frame["target"] = result.spec.target
        feature_frame["display_name"] = result.display_name
        records.append(feature_frame)

    combined = pd.concat(records, ignore_index=True)

    grouped = (
        combined.groupby(["scope", "target", "display_name", "system_group"], observed=True)
        .agg(
            feature_count=("feature", "size"),
            logistic_share=("logistic_share", "sum"),
            forest_share=("forest_share", "sum"),
            correlation_share=("correlation_share", "sum"),
            mean_top_driver_lift=("risk_lift_vs_baseline", "mean"),
        )
        .reset_index()
    )
    grouped["combined_share"] = grouped[["logistic_share", "forest_share", "correlation_share"]].mean(axis=1)

    top_features = (
        combined.sort_values(
            ["scope", "target", "display_name", "system_group", "combined_feature_score", "feature"],
            ascending=[True, True, True, True, False, True],
        )
        .drop_duplicates(subset=["scope", "target", "display_name", "system_group"], keep="first")
        [["scope", "target", "display_name", "system_group", "feature"]]
        .rename(columns={"feature": "top_feature_in_group"})
    )
    grouped = grouped.merge(top_features, on=["scope", "target", "display_name", "system_group"], how="left")
    grouped = grouped.sort_values(["scope", "target", "combined_share"], ascending=[True, True, False]).reset_index(drop=True)
    grouped.to_csv(OUTPUT_DIR / "grouped_variable_importance.csv", index=False)
    return grouped


def compute_segment_volatility(df: pd.DataFrame, features: Iterable[str]) -> pd.DataFrame:
    work = df[["run_id", "MT", "pre_any_defect_50m", "pre_any_defect_200m", "pre_any_defect_500m", "any_defect", *features]].copy()
    work = work.sort_values(["run_id", "MT"]).reset_index(drop=True)

    work["segment"] = np.select(
        [
            work["any_defect"] == 1,
            work["pre_any_defect_50m"] == 1,
            work["pre_any_defect_200m"] == 1,
            work["pre_any_defect_500m"] == 1,
        ],
        ["defect_segment", "pre_defect_50m", "pre_defect_200m", "near_defect_non200m"],
        default="stable_segment",
    )

    records: list[dict[str, float | str]] = []
    for feature in features:
        step_change = work.groupby("run_id", observed=True)[feature].diff().abs()
        summary = pd.DataFrame({"segment": work["segment"], "step_change": step_change}).dropna()
        stable_rate = float(summary.loc[summary["segment"] == "stable_segment", "step_change"].mean())
        pre_50_rate = float(summary.loc[summary["segment"] == "pre_defect_50m", "step_change"].mean())
        pre_200_rate = float(summary.loc[summary["segment"] == "pre_defect_200m", "step_change"].mean())
        defect_rate = float(summary.loc[summary["segment"] == "defect_segment", "step_change"].mean())
        records.append(
            {
                "feature": feature,
                "stable_mean_abs_step": stable_rate,
                "pre_50m_mean_abs_step": pre_50_rate,
                "pre_200m_mean_abs_step": pre_200_rate,
                "defect_mean_abs_step": defect_rate,
                "pre_50m_vs_stable_ratio": pre_50_rate / stable_rate if stable_rate and np.isfinite(stable_rate) else np.nan,
                "pre_200m_vs_stable_ratio": pre_200_rate / stable_rate if stable_rate and np.isfinite(stable_rate) else np.nan,
                "defect_vs_stable_ratio": defect_rate / stable_rate if stable_rate and np.isfinite(stable_rate) else np.nan,
            }
        )

    result = pd.DataFrame(records).sort_values("pre_50m_vs_stable_ratio", ascending=False).reset_index(drop=True)
    result.to_csv(OUTPUT_DIR / "segment_volatility.csv", index=False)
    return result


def draw_volatility_plot(volatility: pd.DataFrame) -> Path:
    plot_df = volatility.head(10).sort_values("pre_50m_vs_stable_ratio")
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.barh(plot_df["feature"], plot_df["pre_50m_vs_stable_ratio"], color="#8a5a44")
    ax.set_title("Pre-defect 50m Volatility Relative To Stable Segments")
    ax.set_xlabel("Mean absolute step-change ratio")
    ax.set_ylabel("")
    fig.tight_layout()
    path = OUTPUT_DIR / "segment_volatility.png"
    fig.savefig(path, dpi=180, bbox_inches="tight")
    plt.close(fig)
    return path


def add_type3_overlay(df: pd.DataFrame) -> tuple[pd.DataFrame, dict[str, int]]:
    audit = pd.read_csv(AUDIT_PATH)
    audit["COIL"] = audit["COIL"].astype("string").str.strip()
    audit["DATE"] = pd.to_datetime(audit["DATE"], errors="coerce").dt.strftime("%Y-%m-%d")
    audit["run_id"] = audit["COIL"].fillna("missing") + "|" + audit["DATE"].fillna("missing")
    audit = audit.loc[(audit["DIF_TIPO_3"] == 1) & (audit["points_in_interval"] > 0)].copy()

    work = df.copy()
    any_type3 = np.zeros(len(work), dtype=np.uint8)
    pre_50 = np.zeros(len(work), dtype=np.uint8)
    pre_200 = np.zeros(len(work), dtype=np.uint8)
    related_500 = np.zeros(len(work), dtype=np.uint8)

    run_indices = work.groupby("run_id", observed=True).indices
    mt_lookup = {run_id: work.loc[idx, "MT"].to_numpy(dtype=float) for run_id, idx in run_indices.items()}

    for row in audit.itertuples(index=False):
        idx = run_indices.get(row.run_id)
        if idx is None:
            continue
        mt = mt_lookup[row.run_id]
        defect_mask = (mt >= float(row.MT_FROM)) & (mt <= float(row.MT_TO))
        pre_50_mask = (mt >= float(row.MT_FROM) - 50.0) & (mt < float(row.MT_FROM))
        pre_200_mask = (mt >= float(row.MT_FROM) - 200.0) & (mt < float(row.MT_FROM))
        pre_500_mask = (mt >= float(row.MT_FROM) - 500.0) & (mt < float(row.MT_FROM))

        any_type3[idx[defect_mask]] = 1
        pre_50[idx[pre_50_mask]] = 1
        pre_200[idx[pre_200_mask]] = 1
        related_500[idx[defect_mask | pre_500_mask]] = 1

    pre_50[any_type3 == 1] = 0
    pre_200[any_type3 == 1] = 0

    work["defect_type3_any_defect"] = any_type3
    work["defect_type3_pre_any_defect_50m"] = pre_50
    work["defect_type3_pre_any_defect_200m"] = pre_200
    work["defect_type3_related_500m"] = related_500

    summary = {
        "type3_any_defect_rows": int(any_type3.sum()),
        "type3_pre_50m_rows": int(pre_50.sum()),
        "type3_pre_200m_rows": int(pre_200.sum()),
        "type3_related_500m_rows": int(related_500.sum()),
    }
    return work, summary


def build_type3_scope(df: pd.DataFrame) -> pd.DataFrame:
    mask = (df["pre_any_defect_500m"] == 0) | (df["defect_type3_related_500m"] == 1)
    return df.loc[mask].copy().reset_index(drop=True)


def run_target_analysis(df: pd.DataFrame, feature_cols: list[str], spec: TargetSpec) -> TargetAnalysis:
    work = build_target_population(df, spec)
    if work[spec.target].sum() == 0:
        raise ValueError(f"No positive rows available for target {spec.target}.")

    train_idx, test_idx = grouped_split(work, spec.target)
    train_df = work.iloc[train_idx].copy()
    test_df = work.iloc[test_idx].copy()
    X_train = train_df[feature_cols]
    X_test = test_df[feature_cols]

    correlation_full, correlation_top = compute_correlations(work, feature_cols, spec.target, spec.output_prefix)
    draw_correlation_plot(correlation_top, TARGET_DESCRIPTIONS[spec.target]["display_name"], spec.output_prefix)
    draw_distribution_plot(
        work,
        correlation_top["feature"].tolist(),
        spec.target,
        TARGET_DESCRIPTIONS[spec.target]["display_name"],
        spec.output_prefix,
    )

    artifacts = fit_models(
        X_train=X_train,
        X_test=X_test,
        y_train=train_df[spec.target],
        y_test=test_df[spec.target],
        feature_cols=feature_cols,
        target=spec.target,
        output_prefix=spec.output_prefix,
    )
    draw_importance_plot(artifacts.driver_table, TARGET_DESCRIPTIONS[spec.target]["display_name"], spec.output_prefix)
    draw_risk_curve_plot(
        X_test,
        test_df[spec.target],
        artifacts.driver_table,
        TARGET_DESCRIPTIONS[spec.target]["display_name"],
        spec.output_prefix,
    )

    return TargetAnalysis(
        spec=spec,
        df=work,
        train_df=train_df,
        test_df=test_df,
        correlation_full=correlation_full,
        correlation_top=correlation_top,
        artifacts=artifacts,
    )


def save_metrics(results: dict[str, TargetAnalysis]) -> pd.DataFrame:
    records: list[dict[str, float | str]] = []
    for result in results.values():
        row: dict[str, float | str] = {
            "scope": result.spec.scope,
            "target": result.spec.target,
            "display_name": result.display_name,
            "population_rows": result.population_rows,
            "positive_rows": result.positive_rows,
            "positive_rate": result.positive_rate,
            "train_rows": len(result.train_df),
            "test_rows": len(result.test_df),
        }
        row.update(result.artifacts.metrics)
        records.append(row)

    metrics_df = pd.DataFrame(records).sort_values(["scope", "target"]).reset_index(drop=True)
    metrics_df.to_csv(OUTPUT_DIR / "model_metrics.csv", index=False)
    return metrics_df


def top_system_for_target(grouped_importance: pd.DataFrame, scope: str, target: str) -> str:
    subset = grouped_importance.loc[(grouped_importance["scope"] == scope) & (grouped_importance["target"] == target)]
    if subset.empty:
        return "n/a"
    return str(subset.sort_values("combined_share", ascending=False).iloc[0]["system_group"])


def build_comparison_table(general_results: dict[str, TargetAnalysis], grouped_importance: pd.DataFrame) -> pd.DataFrame:
    rows: list[dict[str, float | str]] = []
    for target in GENERAL_TARGET_ORDER:
        result = general_results[target]
        metrics = result.artifacts.metrics
        rows.append(
            {
                "target": target,
                "display_name": result.display_name,
                "population_rows": result.population_rows,
                "positive_rows": result.positive_rows,
                "positive_rate": result.positive_rate,
                "logistic_roc_auc": metrics["logistic_roc_auc"],
                "logistic_pr_auc": metrics["logistic_pr_auc"],
                "logistic_lift_top_5pct": metrics["logistic_lift_top_5pct"],
                "logistic_precision_top_5pct": metrics["logistic_precision_top_5pct"],
                "logistic_recall_top_5pct": metrics["logistic_recall_top_5pct"],
                "forest_roc_auc": metrics["forest_roc_auc"],
                "forest_pr_auc": metrics["forest_pr_auc"],
                "forest_lift_top_5pct": metrics["forest_lift_top_5pct"],
                "forest_precision_top_5pct": metrics["forest_precision_top_5pct"],
                "forest_recall_top_5pct": metrics["forest_recall_top_5pct"],
                "top_driver_1": result.artifacts.driver_table.iloc[0]["feature"],
                "top_driver_2": result.artifacts.driver_table.iloc[1]["feature"],
                "top_driver_3": result.artifacts.driver_table.iloc[2]["feature"],
                "dominant_system": top_system_for_target(grouped_importance, "general", target),
                "what_it_captures": TARGET_DESCRIPTIONS[target]["capture_text"],
                "why_useful": TARGET_DESCRIPTIONS[target]["operational_value"],
            }
        )

    comparison = pd.DataFrame(rows)
    comparison.to_csv(OUTPUT_DIR / "comparison_table_targets.csv", index=False)
    return comparison


def dominant_groups_text(grouped_importance: pd.DataFrame, scope: str, target: str, top_n: int = 3) -> str:
    subset = grouped_importance.loc[(grouped_importance["scope"] == scope) & (grouped_importance["target"] == target)]
    if subset.empty:
        return "n/a"
    return ", ".join(subset.sort_values("combined_share", ascending=False)["system_group"].head(top_n).tolist())


def driver_overlap(left: pd.DataFrame, right: pd.DataFrame, top_n: int = 10) -> tuple[int, str]:
    left_features = left["feature"].head(top_n).tolist()
    right_features = right["feature"].head(top_n).tolist()
    overlap = [feature for feature in left_features if feature in set(right_features)]
    return len(overlap), ", ".join(overlap[:6]) if overlap else "none"


def markdown_table(df: pd.DataFrame) -> list[str]:
    cols = list(df.columns)
    header = "| " + " | ".join(cols) + " |"
    rule = "| " + " | ".join(["---"] * len(cols)) + " |"
    rows = [header, rule]
    for _, row in df.iterrows():
        rows.append("| " + " | ".join(str(row[col]) for col in cols) + " |")
    return rows


def write_required_aliases() -> None:
    alias_pairs = [
        ("driver_table_pre_any_defect_50m.csv", "driver_table_pre_50m.csv"),
        ("driver_table_defect_type3_pre_any_defect_50m.csv", "single_defect_driver_table.csv"),
    ]
    for source_name, alias_name in alias_pairs:
        source = OUTPUT_DIR / source_name
        if source.exists():
            shutil.copyfile(source, OUTPUT_DIR / alias_name)


def build_summary_text(
    general_results: dict[str, TargetAnalysis],
    type3_results: dict[str, TargetAnalysis],
    grouped_importance: pd.DataFrame,
) -> str:
    pre_50 = general_results["pre_any_defect_50m"]
    pre_200 = general_results["pre_any_defect_200m"]
    type3_pre_50 = type3_results["defect_type3_pre_any_defect_50m"]
    overlap_count, overlap_features = driver_overlap(pre_50.artifacts.driver_table, type3_pre_50.artifacts.driver_table)

    lines = [
        "Summary",
        "-------",
        (
            f"50m vs 200m: forest PR-AUC is {pre_50.artifacts.metrics['forest_pr_auc']:.3f} at 50m "
            f"vs {pre_200.artifacts.metrics['forest_pr_auc']:.3f} at 200m. The 50m window is the stronger "
            "sparse-label ranking signal, while 200m gives more lead time and higher top-5% lift."
        ),
        (
            f"General vs single-defect: the type 3 pre-50m model shares {overlap_count} of the top 10 drivers "
            f"with the general pre-50m model ({overlap_features})."
        ),
        (
            f"Variable-level vs system-level: the dominant general 50m systems are "
            f"{dominant_groups_text(grouped_importance, 'general', 'pre_any_defect_50m')}, which turns many PLC tags into a smaller operational story."
        ),
    ]
    summary_text = "\n".join(lines)
    (OUTPUT_DIR / "summary_printout.txt").write_text(summary_text, encoding="utf-8")
    print(summary_text)
    return summary_text


def build_report(
    validation: dict,
    comparison_table: pd.DataFrame,
    general_results: dict[str, TargetAnalysis],
    type3_results: dict[str, TargetAnalysis],
    grouped_importance: pd.DataFrame,
    volatility: pd.DataFrame,
    type3_summary: dict[str, int],
) -> Path:
    pre_50 = general_results["pre_any_defect_50m"]
    pre_200 = general_results["pre_any_defect_200m"]
    in_defect = general_results["any_defect"]
    type3_pre_50 = type3_results["defect_type3_pre_any_defect_50m"]
    type3_in_defect = type3_results["defect_type3_any_defect"]

    overlap_50_200_count, overlap_50_200_features = driver_overlap(pre_50.artifacts.driver_table, pre_200.artifacts.driver_table)
    overlap_general_single_count, overlap_general_single_features = driver_overlap(pre_50.artifacts.driver_table, type3_pre_50.artifacts.driver_table)
    top_50_pairs = pre_50.artifacts.risky_pairs.head(5)
    top_50_rules = pre_50.artifacts.top_rules.head(5)
    volatility_top = volatility.head(5)

    comparison_markdown = comparison_table.copy()
    for col in ["positive_rate", "logistic_precision_top_5pct", "logistic_recall_top_5pct", "forest_precision_top_5pct", "forest_recall_top_5pct"]:
        comparison_markdown[col] = comparison_markdown[col].map(pct)
    for col in ["logistic_roc_auc", "logistic_pr_auc", "logistic_lift_top_5pct", "forest_roc_auc", "forest_pr_auc", "forest_lift_top_5pct"]:
        comparison_markdown[col] = comparison_markdown[col].map(lambda value: f"{value:.3f}")
    comparison_markdown = comparison_markdown[
        [
            "display_name",
            "positive_rate",
            "forest_pr_auc",
            "forest_lift_top_5pct",
            "forest_precision_top_5pct",
            "forest_recall_top_5pct",
            "top_driver_1",
            "dominant_system",
        ]
    ].rename(
        columns={
            "display_name": "Target",
            "positive_rate": "Positive Rate",
            "forest_pr_auc": "Forest PR-AUC",
            "forest_lift_top_5pct": "Forest Top-5% Lift",
            "forest_precision_top_5pct": "Forest Top-5% Precision",
            "forest_recall_top_5pct": "Forest Top-5% Recall",
            "top_driver_1": "Top Driver",
            "dominant_system": "Dominant System",
        }
    )

    lines: list[str] = []
    lines.append("# Root Cause Analysis Report")
    lines.append("")
    lines.append("## Key Findings")
    lines.append("")
    lines.append(
        f"- The clean leakage-safe dataset contains {validation['row_count']:,} point-level observations. "
        f"The general positive rates are {pct(pre_50.positive_rate)} for pre-defect 50m, {pct(pre_200.positive_rate)} for pre-defect 200m, "
        f"and {pct(in_defect.positive_rate)} for in-defect rows."
    )
    lines.append(
        f"- The pre-defect 50m target is the preferred warning view in this update: it has the stronger sparse-label forest PR-AUC "
        f"({pre_50.artifacts.metrics['forest_pr_auc']:.3f} vs {pre_200.artifacts.metrics['forest_pr_auc']:.3f} for 200m), "
        f"while the 200m target keeps more lead time and a higher top-5% lift ({pre_200.artifacts.metrics['forest_lift_top_5pct']:.2f}x vs {pre_50.artifacts.metrics['forest_lift_top_5pct']:.2f}x)."
    )
    lines.append(
        f"- Across variable-level and grouped-system views, the most persistent associations sit in "
        f"{dominant_groups_text(grouped_importance, 'general', 'pre_any_defect_50m')}."
    )
    lines.append(
        f"- The type 3 pre-50m model shares {overlap_general_single_count} of the top 10 drivers with the general pre-50m model "
        f"({overlap_general_single_features}), so the single-defect rerun mostly sharpens the existing story rather than replacing it."
    )
    lines.append("")
    lines.append("## Comparison Of Targets")
    lines.append("")
    lines.extend(markdown_table(comparison_markdown))
    lines.append("")
    lines.append("- `any_defect` is strongest for pattern recognition but too late for prevention.")
    lines.append("- `pre_any_defect_200m` is weaker but gives more intervention time.")
    lines.append("- `pre_any_defect_50m` is the best compromise between upstream purity and operational usefulness.")
    lines.append("")
    lines.append("## 50m Vs 200m Early Warning")
    lines.append("")
    lines.append(
        f"- Signal tradeoff: 50m beats 200m on holdout forest PR-AUC ({pre_50.artifacts.metrics['forest_pr_auc']:.3f} vs {pre_200.artifacts.metrics['forest_pr_auc']:.3f}), "
        f"while 200m has the higher top-5% lift ({pre_200.artifacts.metrics['forest_lift_top_5pct']:.2f}x vs {pre_50.artifacts.metrics['forest_lift_top_5pct']:.2f}x)."
    )
    lines.append(
        f"- Driver overlap: the two windows share {overlap_50_200_count} of the top 10 drivers ({overlap_50_200_features}), "
        "so the 50m model mostly sharpens the same upstream signature."
    )
    lines.append("- Practical tradeoff: 50m is stronger for fast response, while 200m is better for earlier lower-confidence monitoring.")
    lines.append("")
    lines.append("## Single Defect Type Analysis")
    lines.append("")
    lines.append(
        f"- Defect type 3 had enough covered data for a focused rerun: {type3_summary['type3_any_defect_rows']:,} in-defect rows, "
        f"{type3_summary['type3_pre_50m_rows']:,} pre-50m rows, and {type3_summary['type3_related_500m_rows']:,} rows in the type 3 related scope."
    )
    lines.append(
        f"- The type 3 pre-50m model reaches forest PR-AUC {type3_pre_50.artifacts.metrics['forest_pr_auc']:.3f}, compared with "
        f"{pre_50.artifacts.metrics['forest_pr_auc']:.3f} for the general pre-50m model."
    )
    lines.append(
        f"- The dominant systems remain {dominant_groups_text(grouped_importance, 'defect_type_3', 'defect_type3_pre_any_defect_50m')}, "
        "which suggests the narrowed defect family reinforces the same main operating systems."
    )
    lines.append(
        f"- The type 3 in-defect contrast remains stronger than the type 3 pre-50m warning signal "
        f"(forest PR-AUC {type3_in_defect.artifacts.metrics['forest_pr_auc']:.3f} vs {type3_pre_50.artifacts.metrics['forest_pr_auc']:.3f}), "
        "which is consistent with symptom-state signals being easier to detect than upstream warning signals."
    )
    lines.append("")
    lines.append("## Variable And System Grouping")
    lines.append("")
    for target in GENERAL_TARGET_ORDER:
        lines.append(
            f"- {TARGET_DESCRIPTIONS[target]['display_name']}: dominant systems are {dominant_groups_text(grouped_importance, 'general', target)}."
        )
    lines.append("- Grouping the variables makes the RCA easier to act on because operations can focus on systems rather than isolated PLC tags.")
    lines.append("")
    lines.append("## Root Cause Explanation")
    lines.append("")
    lines.append("- These models identify variables and regimes that are associated with higher risk. They do not prove physical causation by themselves.")
    lines.append(
        "- The strongest pre-defect pattern is that risk rises when mechanical draw or tension variables, combustion-flow variables, and cooling-related readings move into risky tails together."
    )
    lines.append("- The in-defect target adds symptom-state information, so it is best used as a contrast target rather than the main preventive model.")
    lines.append("")
    lines.append("### High-Risk Combinations On Holdout Data (`pre_any_defect_50m`)")
    lines.append("")
    for _, row in top_50_pairs.iterrows():
        lines.append(
            f"- `{row['rule_a']}` and `{row['rule_b']}` together are associated with a positive rate of {pct(row['positive_rate'])}, "
            f"{row['lift_vs_baseline']:.2f}x the baseline, on {int(row['support']):,} holdout points."
        )
    lines.append("")
    lines.append("### Interpretable Segment Rules (`pre_any_defect_50m`)")
    lines.append("")
    for _, row in top_50_rules.iterrows():
        lines.append(
            f"- `{row['rule']}` is associated with a positive rate of {pct(row['positive_rate'])}, lift {row['lift_vs_baseline']:.2f}x, "
            f"support {int(row['support']):,} points."
        )
    lines.append("")
    lines.append("### Segment Stability And Volatility")
    lines.append("")
    for _, row in volatility_top.iterrows():
        lines.append(
            f"- `{row['feature']}` shows {row['pre_50m_vs_stable_ratio']:.2f}x higher point-to-point volatility in the 50m warning window than in stable segments."
        )
    lines.append("")
    lines.append("## Recommendations")
    lines.append("")
    lines.append("1. Build the operational warning layer around the 50m model first because it has the clearest upstream signal.")
    lines.append("2. Organize troubleshooting by system, starting with the dominant grouped systems and then drilling into the top variables inside them.")
    lines.append("3. Use the 200m model as an earlier lower-confidence drift monitor before the line enters the riskier 50m zone.")
    lines.append("4. Treat the in-defect model as a diagnostic contrast tool, not as the primary action plan.")
    lines.append("5. Pilot a defect-type-specific dashboard for type 3 so the plant can verify which signals are shared vs defect-family-specific.")
    lines.append("")
    lines.append("## Limitations")
    lines.append("")
    lines.append("- This is observational RCA on PLC data. The analysis estimates association with higher risk, not proven causation.")
    lines.append("- Pre-defect models exclude active defect rows on purpose so the warning signatures stay upstream-only.")
    lines.append("- Defect type 3 labels are derived by overlaying covered audit intervals onto the clean point-level dataset.")
    lines.append("- The build dropped 3 defect events with zero production coverage and marked 480 events as partially covered.")
    lines.append("")
    lines.append("## Supporting Files")
    lines.append("")
    lines.append("- `comparison_table_targets.csv`")
    lines.append("- `driver_table_pre_50m.csv`")
    lines.append("- `single_defect_driver_table.csv`")
    lines.append("- `grouped_variable_importance.csv`")
    lines.append("- `model_metrics.csv`")
    lines.append("- `segment_volatility.csv`")

    report_path = OUTPUT_DIR / "rca_root_cause_report.md"
    report_path.write_text("\n".join(lines), encoding="utf-8")
    return report_path


def main() -> dict[str, str | int]:
    ensure_output_dir()
    sns.set_theme(style="whitegrid")

    df, feature_cols = load_dataset(DATA_PATH)
    validation = json.loads(VALIDATION_PATH.read_text(encoding="utf-8"))

    general_results = {spec.target: run_target_analysis(df, feature_cols, spec) for spec in GENERAL_SPECS}

    top_volatility_features = sorted(
        {
            *general_results["pre_any_defect_50m"].artifacts.driver_table["feature"].head(8).tolist(),
            *general_results["pre_any_defect_200m"].artifacts.driver_table["feature"].head(8).tolist(),
        }
    )
    volatility = compute_segment_volatility(df, features=top_volatility_features)
    draw_volatility_plot(volatility)

    df_type3, type3_summary = add_type3_overlay(df)
    df_type3_scope = build_type3_scope(df_type3)
    type3_results = {spec.target: run_target_analysis(df_type3_scope, feature_cols, spec) for spec in TYPE3_SPECS}

    all_results = {**general_results, **type3_results}
    grouped_importance = build_grouped_importance(all_results)
    comparison_table = build_comparison_table(general_results, grouped_importance)
    metrics_df = save_metrics(all_results)
    write_required_aliases()

    report_path = build_report(
        validation=validation,
        comparison_table=comparison_table,
        general_results=general_results,
        type3_results=type3_results,
        grouped_importance=grouped_importance,
        volatility=volatility,
        type3_summary=type3_summary,
    )
    summary_text = build_summary_text(general_results, type3_results, grouped_importance)

    summary = {
        "row_count": len(df),
        "feature_count": len(feature_cols),
        "general_pre_50m_top_driver": general_results["pre_any_defect_50m"].artifacts.driver_table.loc[0, "feature"],
        "general_pre_200m_top_driver": general_results["pre_any_defect_200m"].artifacts.driver_table.loc[0, "feature"],
        "general_any_defect_top_driver": general_results["any_defect"].artifacts.driver_table.loc[0, "feature"],
        "single_defect_top_driver": type3_results["defect_type3_pre_any_defect_50m"].artifacts.driver_table.loc[0, "feature"],
        "report_path": str(report_path),
        "metrics_path": str(OUTPUT_DIR / "model_metrics.csv"),
        "comparison_table_path": str(OUTPUT_DIR / "comparison_table_targets.csv"),
        "grouped_importance_path": str(OUTPUT_DIR / "grouped_variable_importance.csv"),
        "summary_text_path": str(OUTPUT_DIR / "summary_printout.txt"),
    }
    (OUTPUT_DIR / "run_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
    print(json.dumps(summary, indent=2))
    print(metrics_df.to_string(index=False))
    print(summary_text)
    return summary



## Run The RCA

Execute the next cell to run the full analysis pipeline and regenerate the report plus all supporting outputs.


In [ ]:
summary = main()
summary
